In [ ]:
import csv
import re
import time
import random
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Union

import requests
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
ROOT = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\4 - RQ4\2nd_Try")

TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

# IMPORTANT: no leading "\" here (otherwise it becomes an absolute path on Windows)
URL_LIST_CSV = ROOT / "URL_List_Instru.csv"
EPISODES_CSV = ROOT / "List_Change_Episodes.csv"

OUT_DIR = ROOT / "RouteA_EpisodeRunMetrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- NEW: stop after collecting K instrumentation-related CI records per episode ---
K_TARGET_RECORDS_PER_EPISODE = 100

# Safety cap: maximum commits scanned per episode (prevents pathological long scans)
MAX_COMMITS_SCAN_PER_EPISODE = 5000

# If True, skip GitHub Actions check-runs (keeps only external CI apps that report checks)
EXCLUDE_GITHUB_ACTIONS_CHECKRUNS = False  # set True if you ONLY want non-GHA providers

# Retry/backoff
CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

# ============================================================
# Instrumentation-ish matching (tune as needed)
# ============================================================
INSTRU_NAME_RE = re.compile(
    r"(androidtest|connectedandroidtest|connectedcheck|devicecheck|manageddevice|gmd|"
    r"instrument(ation)?|am\s+instrument|espresso|uiautomator|emulator|firebase\s+test|"
    r"test\s*lab|device\s*farm|browserstack|saucelabs|kobiton|appcenter|flank|maestro|marathon|spoon|baselineprofile|benchmark)",
    re.IGNORECASE,
)

# ============================================================
# Helpers
# ============================================================
def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def parse_repo_full_name(repo_url_or_fullname: str) -> Optional[str]:
    s = (repo_url_or_fullname or "").strip()
    if not s:
        return None
    if re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", s):
        return s
    m = re.search(r"github\.com[:/]+([A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+)", s, re.IGNORECASE)
    if m:
        full = m.group(1)
        return full[:-4] if full.endswith(".git") else full
    return None

def read_repo_list(csv_path: Path) -> List[str]:
    rows = list(csv.reader(csv_path.open("r", encoding="utf-8", errors="ignore", newline="")))
    if not rows:
        return []
    header = [c.strip().lower() for c in rows[0]]
    has_header = any(("url" in c or "repo" in c or "full" in c) for c in header)
    col_idx = 0
    if has_header:
        for i, name in enumerate(header):
            if name in ("url", "repo_url", "repo", "repository", "full_name"):
                col_idx = i
                break
    start = 1 if has_header else 0
    out = []
    for r in rows[start:]:
        if not r or col_idx >= len(r):
            continue
        full = parse_repo_full_name((r[col_idx] or "").strip())
        if full:
            out.append(full)
    return sorted(set(out))

def seconds_between(a: Optional[str], b: Optional[str]) -> Optional[int]:
    try:
        if not a or not b:
            return None
        da = pd.to_datetime(a, utc=True)
        db = pd.to_datetime(b, utc=True)
        sec = int((db - da).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None

def is_instru(name_or_context: str) -> bool:
    return bool(INSTRU_NAME_RE.search(name_or_context or ""))

# ============================================================
# GitHub client
# ============================================================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "routeA-episode-run-metrics/1.1",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        time.sleep(max(1, soonest - now + 2))

    def _backoff(self, attempt: int) -> None:
        time.sleep(min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random())

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code
            try:
                st.remaining = int(resp.headers.get("X-RateLimit-Remaining", "0"))
            except Exception:
                pass
            try:
                st.reset_epoch = int(resp.headers.get("X-RateLimit-Reset", "0"))
            except Exception:
                pass

            if resp.status_code == 404:
                return None

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and ("rate limit" in text_l or "secondary rate limit" in text_l):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if not data:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1

# ============================================================
# GitHub endpoints
# ============================================================
def load_tokens_from_env_file(env_path: Path) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}")
    return tokens

def get_repo_default_branch(gh: GitHubClient, full_name: str) -> str:
    url = f"https://api.github.com/repos/{full_name}"
    data = gh.request_json("GET", url)
    if not data or not isinstance(data, dict):
        return ""
    return (data.get("default_branch") or "").strip()

def list_commit_shas_in_window(
    gh: GitHubClient,
    full_name: str,
    branch: str,
    since_iso: str,
    until_iso: str,
) -> Iterable[str]:
    url = f"https://api.github.com/repos/{full_name}/commits"
    params = {"sha": branch, "since": since_iso, "until": until_iso}
    for c in gh.paginate(url, params=params, item_key=""):
        sha = c.get("sha")
        if isinstance(sha, str) and len(sha) >= 7:
            yield sha

def list_check_runs_for_commit(gh: GitHubClient, full_name: str, sha: str) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/commits/{sha}/check-runs"
    data = gh.request_json("GET", url)
    if not data or not isinstance(data, dict):
        return []
    return data.get("check_runs", []) or []

def get_combined_status_for_commit(gh: GitHubClient, full_name: str, sha: str) -> Dict:
    url = f"https://api.github.com/repos/{full_name}/commits/{sha}/status"
    data = gh.request_json("GET", url)
    return data if isinstance(data, dict) else {}

# ============================================================
# Episodes loader
# ============================================================
def load_episodes(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    cols_l = {c.lower(): c for c in df.columns}

    def pick(*names):
        for n in names:
            if n in cols_l:
                return cols_l[n]
        return None

    repo_c  = pick("full_name", "repo", "repository")
    start_c = pick("episode_start_utc", "start_utc", "episode_start")
    end_c   = pick("episode_end_utc", "end_utc", "episode_end")
    style_c = pick("episode_env_styles", "env_styles", "exec_env_style", "style")
    eid_c   = pick("episode_id", "id", "episode")

    if not repo_c or not start_c or not end_c:
        raise ValueError(f"Could not find repo/start/end columns in {path.name}. Columns={list(df.columns)}")

    out = df.copy()
    out["full_name"] = out[repo_c].astype(str).str.strip()
    out["episode_start_utc"] = pd.to_datetime(out[start_c], utc=True, errors="coerce")
    out["episode_end_utc"]   = pd.to_datetime(out[end_c], utc=True, errors="coerce")
    out["episode_env_styles"] = out[style_c].astype(str) if style_c else ""
    out["episode_id"] = out[eid_c] if eid_c else range(len(out))
    out = out.dropna(subset=["full_name", "episode_start_utc", "episode_end_utc"])
    return out[["full_name", "episode_id", "episode_start_utc", "episode_end_utc", "episode_env_styles"]]

# ============================================================
# Main
# ============================================================
def main():
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH)
    gh = GitHubClient(tokens)

    repos = set(read_repo_list(URL_LIST_CSV))
    episodes = load_episodes(EPISODES_CSV)
    episodes = episodes[episodes["full_name"].isin(repos)].copy()

    print(f"[load] repos={len(repos)}")
    print(f"[load] episodes(in studied repos)={len(episodes)}")

    default_branch_cache: Dict[str, str] = {}
    raw_rows = []

    for full_name, grp in episodes.groupby("full_name"):
        if full_name not in default_branch_cache:
            default_branch_cache[full_name] = get_repo_default_branch(gh, full_name)
        branch = default_branch_cache[full_name]
        if not branch:
            continue

        for _, ep in grp.iterrows():
            since_iso = ep["episode_start_utc"].isoformat().replace("+00:00", "Z")
            until_iso = ep["episode_end_utc"].isoformat().replace("+00:00", "Z")

            collected = 0
            commits_scanned = 0

            # Stream commits; stop when we collect K records or hit commit safety cap
            for sha in list_commit_shas_in_window(gh, full_name, branch, since_iso, until_iso):
                commits_scanned += 1
                if commits_scanned > MAX_COMMITS_SCAN_PER_EPISODE:
                    break
                if collected >= K_TARGET_RECORDS_PER_EPISODE:
                    break

                # 1) Check runs
                check_runs = list_check_runs_for_commit(gh, full_name, sha)
                for cr in check_runs:
                    if collected >= K_TARGET_RECORDS_PER_EPISODE:
                        break

                    app = cr.get("app") or {}
                    app_slug = (app.get("slug") or "").strip().lower()
                    if EXCLUDE_GITHUB_ACTIONS_CHECKRUNS and app_slug == "github-actions":
                        continue

                    name = (cr.get("name") or "").strip()
                    if not is_instru(name):
                        continue

                    started_at = cr.get("started_at") or ""
                    completed_at = cr.get("completed_at") or ""
                    dur = seconds_between(started_at, completed_at)
                    concl = cr.get("conclusion") or cr.get("status") or ""

                    raw_rows.append({
                        "full_name": full_name,
                        "default_branch": branch,
                        "episode_id": ep["episode_id"],
                        "episode_env_styles": ep["episode_env_styles"],
                        "episode_start_utc": since_iso,
                        "episode_end_utc": until_iso,
                        "commit_sha": sha,
                        "record_type": "check_run",
                        "provider": app_slug or "unknown_app",
                        "job_name": name,
                        "conclusion": concl,
                        "started_at": started_at,
                        "completed_at": completed_at,
                        "duration_seconds": dur if dur is not None else "",
                        "html_url": cr.get("html_url") or "",
                        "collected_at_utc": now_utc_iso(),
                        "run_instance_key": f"checkrun:{cr.get('id')}",
                    })
                    collected += 1

                if collected >= K_TARGET_RECORDS_PER_EPISODE:
                    break

                # 2) Commit statuses
                st = get_combined_status_for_commit(gh, full_name, sha)
                statuses = st.get("statuses", []) if isinstance(st, dict) else []
                for s in statuses:
                    if collected >= K_TARGET_RECORDS_PER_EPISODE:
                        break

                    context = (s.get("context") or "").strip()
                    if not is_instru(context):
                        continue

                    state = (s.get("state") or "").strip().lower()
                    created_at = s.get("created_at") or ""
                    updated_at = s.get("updated_at") or ""
                    dur = seconds_between(created_at, updated_at)

                    prov = (context.split("/")[0] if "/" in context else context.split(":")[0]).strip().lower()

                    raw_rows.append({
                        "full_name": full_name,
                        "default_branch": branch,
                        "episode_id": ep["episode_id"],
                        "episode_env_styles": ep["episode_env_styles"],
                        "episode_start_utc": since_iso,
                        "episode_end_utc": until_iso,
                        "commit_sha": sha,
                        "record_type": "commit_status",
                        "provider": prov,
                        "job_name": context,
                        "conclusion": state,
                        "started_at": "",
                        "completed_at": "",
                        "duration_seconds": dur if dur is not None else "",
                        "html_url": s.get("target_url") or "",
                        "collected_at_utc": now_utc_iso(),
                        "run_instance_key": f"status:{sha}:{context}:{created_at}",
                    })
                    collected += 1

            # Optional progress print per episode
            print(f"[episode] {full_name} ep={ep['episode_id']} collected={collected}/{K_TARGET_RECORDS_PER_EPISODE} commits_scanned={commits_scanned}")

    raw_df = pd.DataFrame(raw_rows)
    raw_out = OUT_DIR / "routeA_runs_labeled.csv"
    raw_df.to_csv(raw_out, index=False, encoding="utf-8-sig")
    print(f"[save] {raw_out} rows={len(raw_df)}")

    if len(raw_df) == 0:
        print("[warn] No instrumentation-related checks/statuses found in those episode windows.")
        print("Likely causes: CI didn’t report to GitHub, or job names don’t match INSTRU_NAME_RE.")
        return

    # Normalize success/failure
    df = raw_df.copy()
    c = df["conclusion"].astype(str).str.lower()

    # check_run conclusions commonly: success, failure, cancelled, timed_out, neutral, skipped, action_required, etc.
    # commit_status states commonly: success, failure, error, pending
    df["is_success"] = c.isin({"success", "neutral", "skipped"})
    df["is_failure"] = c.isin({"failure", "error", "cancelled", "timed_out", "action_required"})
    df["duration_s"] = pd.to_numeric(df["duration_seconds"], errors="coerce")

    # (A) Repo × Episode(style) × Provider × Job metrics
    gcols = ["full_name", "episode_id", "episode_env_styles", "provider", "job_name"]
    wf_ep = df.groupby(gcols).agg(
        runs=("run_instance_key", "nunique"),
        success_runs=("is_success", "sum"),
        failure_runs=("is_failure", "sum"),
        success_rate=("is_success", "mean"),
        dur_mean_s=("duration_s", "mean"),
        dur_median_s=("duration_s", "median"),
        dur_p95_s=("duration_s", lambda x: x.dropna().quantile(0.95) if x.dropna().size else float("nan")),
    ).reset_index()

    wf_ep_out = OUT_DIR / "routeA_workflow_episode_metrics.csv"
    wf_ep.to_csv(wf_ep_out, index=False, encoding="utf-8-sig")
    print(f"[save] {wf_ep_out} rows={len(wf_ep)}")

    # (B) Style overall metrics
    style = df.groupby(["episode_env_styles"]).agg(
        runs=("run_instance_key", "nunique"),
        success_runs=("is_success", "sum"),
        failure_runs=("is_failure", "sum"),
        success_rate=("is_success", "mean"),
        dur_mean_s=("duration_s", "mean"),
        dur_median_s=("duration_s", "median"),
    ).reset_index()

    style_out = OUT_DIR / "routeA_style_overall_metrics.csv"
    style.to_csv(style_out, index=False, encoding="utf-8-sig")
    print(f"[save] {style_out} rows={len(style)}")

    print("Done.")

if __name__ == "__main__":
    main()
